# Project NYPD Enforcement Gap
##### Team Members: Haeji Ahn, Amy Lin, Amanda Song

## 1. Introduction

The New York Police Department (NYPD) is the largest municipal police department in the United States, with a huge operating budget that is greater than the combined budgets of several other agencies in New York City. Our project aims to study enforcement patterns, particularly when a stop, question, and frisk (SQF) action is undertaken by the police, and identify what kinds of stops lead to physical escalation by the police. We aim to answer the question: To what extent do demographic characteristics (e.g, race, age, and sex) and situational factors (e.g., location, why the stop was initiated, and the suspected crime) predict the use of physical force during a police encounter?

By identifying which factors are correlated with an increased risk of use of physical force, we can increase transparency regarding the NYPD's enforcement patterns, exposing systemic institutional biases that may exist. 

## 2. Data

Our dataset is an Excel file from the NYPD's SQF database, containing detailed information on all documented pedestrian stops made by NYPD officers based on the NYPD UF-250 form that they fill out post-stop. We specifically focused on the most recent dataset from 2025, which has 26,422 records. 

We used the following features:
1. "SUSPECT_RACE_DESCRIPTION": Description of individual's race
2. "SUSPECT_REPORTED_AGE": Reported age of individual
3. "SUSPECT_SEX": Sex of individual
4. "STOP_WAS_INITIATED": Whether a stop was initiated based on C/W on scene, from a radio run, or self-initiated
5. "SUSPECTED_CRIME_DESCRIPTION": Crime that the individual was suspected to have committed
6. "STOP_LOCATION_PATROL_BORO_NAME": One of 8 police patrol boroughs where the stop occurred (e.g., Manhattan North, Manhattan South, Bronx)

The feature we extracted is whether an individual experienced use of physical force by the police officer, and is our output feature. To do so, we combined "PHYSICAL_FORCE_CEW_FLAG", "PHYSICAL_FORCE_DRAW_POINT_FIREARM_FLAG", "PHYSICAL_FORCE_HANDCUFF_SUSPECT_FLAG", "PHYSICAL_FORCE_OC_SPRAY_USED_FLAG", "PHYSICAL_FORCE_OTHER_FLAG", "PHYSICAL_FORCE_RESTRAINT_USED_FLAG", and "PHYSICAL_FORCE_WEAPON_IMPACT_FLAG" to produce a binary feature that indicated if physical force was used.

We conducted comprehensive data cleaning and preparation to ensure that our dataset was prepared for model training and analysis. Key pre-processing steps included:
1. Replacing values that had the string "(null)" with actual NaN values
2. Converting age values to numeric format while coercing any invalid entries
3. Dropping rows with NaN values in race, age, sex, and location features -- These were missing values from the dataset.
4. Dropping records where suspects' ages were below 12 -- These were likely data entry errors.
5. Aggregating ages into broader categories (<18, 18–24, 25–44, 45–54, >=55) -- The age classifications were referenced from a report from the NYC Mayor's Office of Criminal Justice on youth arrests and recidivism.
6. Converting missing values in physical force columns to 0 while concurrently converting "Y" values to 1 -- These missing values indicated that no physical force had been applied, rather than representing true missing data.

To further prepare the data for modelling, we converted categorical features into numerical format using one-hot encoding and dropped one baseline category for each feature to avoid multicollinearity. We also aggregated the multiple physical force indicator columns into a single binary target feature which represented whether any physical force was used by the NYPD during the interaction.  

One limitation of our dataset is that it is slightly imbalanced, with approximately 80% negative labels and 20% positive labels. We addressed this imbalance through various methods for each of our models and by prioritizing certain metrics such as recall and F1 score over accuracy, which will be discussed in subsequent sections. Additionally, the dataset did not have a corresponding data dictionary, which required us to infer the meanings of certain abbreviations through further desktop research. For example, under the "STOP_WAS_INITIATED" feature, there was a value "Based on C/W on Scene", which we interpreted as whether there was a "criminal weapon" on scene. Although we attempted to obtain official data definitions from the NYPD, we unfortunately did not receive a response.

## 3. Models and Results

We trained and evaluated our dataset using three model classes:
1. Linear model: Logistic regression
2. Non-linear model: Decision trees and random forests
3. Neural network model

Prior to training, we split our dataset into training and testing sets using scikit-learn's *train_test_split* to ensure that we had unbiased data to evaluate our model's performance post-training and tuning. We also set seeds across our models to ensure reproducibility of our analysis and results.

Based on our research question on predicting the use of force by the NYPD, we found it especially important to minimize false negatives, as failing to identify cases where physical force is used may understate instances where certain demographic groups are disproportionately affected. As such, we prioritized recall in our models, where we selected optimal hyperparameter combinations which had the highest recall. In contrast, while accuracy is an evaluation metric that is commonly used in machine learning, this was not our focus due to the moderate class imbalance in our dataset. 

Of our models, xxx was the best-performing model with a recall score of xxx on the test set. This was likely the case as ...
xxx was the worst-performing model, likely due to ...

### 3.1. Linear Model: Logistic Regression

#### 3.1.1. Model Set-up and Hyperparameter Tuning

We decided to use logistic regression to train our linear model. Logistic regression enabled us to identify variables that are more strongly associated with the probability of physical force and predict a binary outcome based on the given demographic and situational features, especially since our data is not perfectly linearly separable. 

We initialized our model using scikit-learn's *LogisticRegression* class, where we set *solver="liblinear"* to allow for both L1 and L2 regularization, and set *class_weight="balanced"* to assign higher weights to the minority class during model training given the imbalance of our dataset. To prepare for cross-validation during the training process, we defined how our training set should be split by using scikitlearn's *KFold*, implementing 5-fold cross-validation in subsequent steps.

Key hyperparameters that we wanted to tune were mainly related to regularization:
1. **Penalty type**: L1 or L2
2. **C**: Inverse of penalty strength (1.0, 0.1, 0.01, 0.001, 0.0001)  

As our model contains a sizable number of features, we aimed to effectively control its complexity and prevent overfitting, which made tuning the above hyperparameters important. We defined a parameter grid that included the hyperparameters we wanted to tune and specified multiple evaluation metrics (accuracy, precision, recall, F1). Using the cross-validation split defined earlier, we passed these into scikit-learn's *GridSearchCV* and fit the search on our training data to find the optimal hyperparameter combination. As mentioned, we prioritized recall, thus we set *refit="recall"* such that the best hyperparameter combination returned would correspond to the one that yielded the highest recall score. The optimal model returned had a C = 0.01 with L1 regularization, achieving a recall score of 0.6571.

#### 3.1.2. Estimated Weights and Bias

Using the tuned hyperparameters from the optimal model, we obtained the estimated coefficients and bias term for our classifier. As L1 regularization was applied, many feature weights were set to zero, leaving only nine features with non-zero weights that were identified as having the highest importance to our model's predictions of the use of force by the NYPD. Among these, the features with the largest positive weights (indicating a higher likelihood of force being used) were whether the stop occurred in Manhattan South (0.4958) and whether the suspect was male (0.1550). On the other hand, the features with the largest negative weights (indicating a lower likelihood of force being used) were whether the stop was self-initiated (-0.5504) and whether the stop was in the Bronx (-0.3073). 

We had previously expected racial features to play a larger role in predicting outcomes. However, only one racial feature -- whether the suspect was identified as Black -- had a non-zero weight, and its weight was rather small albeit positive (0.0369). A possible explanation is that some racial effects may have been absorbed into correlated geographic or situational features, such as patrol boroughs, due to multicollinearity between features. 

#### 3.1.3. Testing and Evaluation

We applied the tuned hyperparameters from the optimal model to our testing set by calling *predict(X_test)* on the fitted grid_search model. Using scikit-learn's *classification_report* to compare our predictions against actual labels in the testing set, we evaluated key evaluation metrics, including recall, precision, F1 score, and accuracy. The model achieved a recall score of 0.68 on our test set, which was slightly higher than the recall score obtained during validation. The F1 score, precision, and accuracy were 0.41, 0.30, and 0.59 respectively.

Overall, while our model was able to correctly identify a meaningful number of positives in our dataset which was in line with our goal to minimize false negatives, its performance definitely had room for improvement. The relatively low scores across all metrics returned from the testing set suggested that a linear model may not have been sufficiently flexible to capture more complex patterns and relationships in our dataset. 

#### INSERT CLASSIFICATION TABLE AND CONFUSION MATRIX

### 3.2. Non-linear Model: Decision Trees and Random Forest

xxx

### 3.3. Neural Network Model

xxx

## 4. Learnings and Challenges

Did anything unexpected happen? What parts of the project were surprisingly difficult? 


Talk about how there were many hyperparameter combinations that we wanted to tune, but learnt that we had to strategically select and trim hyperparamter combinations to validate due to the computational power and time required if there were too many different combinations

## 5. Conclusion

What did you learn about your research question? What did you learn about machine learning through the tasks?  How would you continue this research if you wanted to further investigate? If you had more time and infinite resources, how would you continue your project?